# RetNet-300M: Latent Reasoning (Coconut curriculum)

Starts from the pretrained checkpoint in `runs/retnet-300m/` and teaches the model to reason in latent space:
after `<bot>` it feeds its own hidden state back as input for `n` steps, then `<eot>` switches back to text.

* **Stage 0**: plain step-by-step fine-tuning on GSM8K (the explicit chain-of-thought baseline).
* **Stage s**: the first `s` reasoning steps are replaced by `s × c` continuous thoughts.
* Each stage is evaluated on GSM8K test with that many thoughts. Every latent step is one recurrent update of a
  fixed-size state, so thinking longer costs no extra memory.

Resumable: Run all continues from the last checkpoint in `runs/retnet-300m-latent/`.

In [ ]:
# 1. GPU, Drive, code
import torch
assert torch.cuda.is_available(), "Runtime -> Change runtime type -> GPU"
print(torch.cuda.get_device_name(0))
from google.colab import drive
drive.mount("/content/drive")
import os, pathlib, subprocess, sys, shutil, json

FOLDER_NAME = "retnet-300m"
MOUNT = pathlib.Path("/content/drive")
candidates = [MOUNT / "MyDrive" / FOLDER_NAME]
for root in ("Shareddrives", ".shortcut-targets-by-id"):
    if (MOUNT / root).is_dir():
        candidates += [d / FOLDER_NAME for d in sorted((MOUNT / root).iterdir()) if d.is_dir()]
DRIVE = next((c for c in candidates if c.is_dir()), MOUNT / "MyDrive" / FOLDER_NAME)
BUNDLE, RUNS = DRIVE / "retnet-300m-colab.zip", DRIVE / "runs"
assert BUNDLE.is_file(), f"Could not find {BUNDLE}"
CODE = pathlib.Path("/content/retnet-300m")
shutil.rmtree(CODE, ignore_errors=True)
subprocess.run(["unzip", "-oq", str(BUNDLE), "-d", str(CODE)], check=True)
assert (CODE / "scripts" / "train_latent.py").is_file(), "Bundle is outdated: rebuild with scripts/make_colab_bundle.sh and re-upload."
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "tokenizers", "datasets"], check=True)
os.chdir(CODE)
PY = sys.executable

def run(cmd):
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"command failed (exit {p.returncode}): {cmd}")
print((CODE / "BUILD_INFO.txt").read_text().strip())

In [ ]:
# 2. Settings
INIT = RUNS / "retnet-300m"          # pretrained run (needs a ckpt_step_*.pt and tokenizer.json)
OUT = RUNS / "retnet-300m-latent"
MAX_STAGE = 4        # stage s replaces the first s steps; GSM8K solutions have ~3-4 steps
C = 2                # thoughts per replaced step -> final stage thinks for MAX_STAGE * C steps
EPOCHS_STAGE0 = 3    # explicit CoT fine-tuning
EPOCHS_PER_STAGE = 2
METAMATH = 0         # extra MetaMathQA GSM problems (0 = GSM8K train only, ~2.5-3 h on a T4; 20000 adds ~3x)

from retnet.checkpoint import checkpoint_candidates
assert checkpoint_candidates(INIT), f"No checkpoint in {INIT}. Rename ckpt_step_0015082.pt.recovered to ckpt_step_0015082.pt."
print("init:", [p.name for p in checkpoint_candidates(INIT)][:2])
print("latent run:", [p.name for p in checkpoint_candidates(OUT)][:2] or "not started")

In [ ]:
# 3. Train the curriculum (safe to stop; Run all resumes)
run(f"{PY} -u scripts/train_latent.py --init '{INIT}' --out '{OUT}' --max_stage {MAX_STAGE} --c {C} "
    f"--epochs_stage0 {EPOCHS_STAGE0} --epochs_per_stage {EPOCHS_PER_STAGE} --metamath {METAMATH} --resume")

In [ ]:
# 4. Results per stage, then accuracy vs. number of thoughts on the full GSM8K test set
recs = [json.loads(l) for l in open(OUT / "log.jsonl") if '"eval"' in l]
for r in recs:
    e = r["eval"]
    print(f"stage {r['stage']}: {e['n_latent']:2d} thoughts -> GSM8K {100 * e['acc']:.1f}% +/- {100 * e['stderr']:.1f}% ({e['n']} problems)")
SWEEP = ",".join(str(n) for n in range(0, MAX_STAGE * C + 5, 2))
run(f"{PY} -u scripts/eval_latent.py --run '{OUT}' --n_latent {SWEEP}")

In [ ]:
# 5. Try it
from retnet.checkpoint import load_run, load_latest_checkpoint
from retnet.latent import LatentReasoner
model, tok, step = load_run(OUT, "cuda")
lat = load_latest_checkpoint(OUT, "cpu")[0]["latent"]
reasoner = LatentReasoner(model, lat["scale"]).cuda().eval()
Q = "A baker bakes 7 trays of 24 muffins and sells 150. How many are left?"
for n in (0, lat["n_latent"]):
    text, calls = reasoner.generate(tok, [Q], n)[0]
    print(f"--- {n} thoughts ({calls} calculator calls):\n{text.strip()}\n")